# 타이타닉 분석

# 분석 목적
# 타이타닉 생존 여부에 영향을 미친 주요 요인에 대한 제계적 분석


In [ ]:
# 1. 데이터 수집
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

from google.colab import drive
drive.mount('/content/drive')
df_train = pd.read_csv('/content/drive/My Drive/Colab Notebooks/titanic/train.csv')

#엑셀로 저장
df_train.to_excel("titanic.xlsx")
#df_train

![alt text](image.png)

In [ ]:
# 2. 전처리
# 2-1) 구조진단
df_train.info()

# 분석 = 인사이트
# 컬럼명을 한글로 분석완료 하였음
# 3개의 컬럼에서 결측치 보인다.
# 총 13 개의 컬럼에서 891 행의 데이터가 있다.

df_train.head(100)
# 분석 
# SibSp	Parch => 해당 데이터는 좀더 분석 필요
df_train.shape
# 분석 
# (891, 12) 13 개의 컬럼 891명의 개인 정보가 담겨져 있음



In [ ]:
!pip install missingno 

In [ ]:
import missingno as msno

# 2-2) 정제 (결측치, 이상치)

df_train.isnull().sum()
msno.matrix(df_train) 

missing_pct = (df_train.isnull().sum() / len(df_train) * 100).sort_values(ascending=False)
print(missing_pct)

# 1)순수 데이터 분석만 할시는 데이터 정제 및 결측치 이상치 처리할 필요 없음

# Age 결측치 처리
# 20%면 버리기 아까움. = 결측이 너무 많지는 않음
# 판단 => 채우는 방향으로 
# Age 연속형 데이타
# 숫자는 => 평균 ,중앙값, 최빈값, 최대최소 , 예측값으로 결측값을 채울수 있음
df_train["Age"].skew() #왜도 체크 값(0.38)
# 왜도 체크 => 분포가 약간 치우침
# 예시
# 0.38
# → 약한 우측 왜도 => 왜도가 있다는 말은 10, 20, 30, 40, 300 왼쪽으로 끌리는 데이터가 있다는말 임
# 평균보다 중앙값 안정.

# | 왜도 값       | 해석      |
# | ---------- | ------- |
# | -0.5 ~ 0.5 | 거의 대칭   |
# | ±0.5 ~ ±1  | 약간 치우침  |
# | ±1 이상      | 강하게 치우침 |

# | 항목   | 왜도 (Skewness)   | 첨도 (Kurtosis)       |
# | ----  | ---------------   | -------------------- |
# | 질문   | 데이터가 한쪽으로 치우쳤나? | 데이터가 뾰족하거나 꼬리가 두꺼운가? |
# | 보는 것 | 좌우 비대칭         | 중심 집중 + 꼬리 두께        |
# | 관심   | 방향                | 극단값(이상치)             |
# | 기준값  | 0                 | 0 (초과첨도 기준)          |
# | 영향   | 평균 이동           | 이상치 증가               |

df_train["Age"] = df_train["Age"].fillna(df_train["Age"].median())


# Cabin: => 77%
# 결측률이 너무큼 
# 채워도 의미 없음
# 해당행 제거시 더 큰 위험 => 데이터가 망가짐
# 제거 고려

# 결측률	판단
# < 10%	대체
# 10~30%	상황 판단
# 30~50%	신중
# >50%	제거 고려

df_train.drop(columns=["cabin"],errors="ignore",inplace=True)

# Embarked 처리
# 2 / 891 = 결측률 0.2%
# 범주형 데이터
# 최빈값으로 대체

df_train["Embarked"] = df_train["Embarked"].fillna(df_train["Embarked"].mode()[0])

# 전처리 판단기준
#1.결측률 체크
#2.컬럼 타입 체크
#3.데이타 분포
#4.채우면 왜곡 되는가?
#5.제거하면 손실이 큰가?

# 분석, 인사이트 발견
# Age
# → 숫자형 + 20%
# → 중앙값

# Cabin
# → 77%
# → 컬럼 제거

# Embarked
# → 범주형 + 0.2%
# → 최빈값


In [ ]:
# 2-3) 문자열 및 시계열 처리

# 예제)
# 1.이름이 생존률과 차이및 관계가 있을까?
# 2.예측에 도움이 되는 정보는 아님
# 3.범주형이 아니어서 그룹별로 취할수 없는 정보가 없음

# | Name                       | Title |
# | -------------------------- | ----- |
# | Braund, Mr. Owen Harris    | Mr    |
# | Cumings, Mrs. John Bradley | Mrs   |
# | Heikkinen, Miss. Laina     | Miss  |

# Braund, Mr. Owen Harris
#          ↑
# 추출 결과
# Mr
df_train["Title"] = df_train["Name"].str.extract(" ([A-Za-z]+)\.")
# | Title  | 의미    |
# | ------ | ----- |
# | Mr     | 성인 남성 |
# | Mrs    | 기혼 여성 |
# | Miss   | 미혼 여성 |
# | Master | 어린 남자 |

# Name 그대로 쓰지 않은이유
# 1.모든 이름이 유일함
# 2.데이터 종류 너무 많아짐
# 3.일반화가 어려워짐

df_train

In [ ]:
# 2-4) 정리 및 변환
# 불필요한 컬럼 제거
# 단순한 컬럼수 줄이기는 아님
# 예측및 분석에 도움이 되는 정보인지 체크. 예) 중복인가, 식별자인가?
# 를 판단해서 제거

# PassengerId
# 승객번호 1번이라 생존했나?
# 승객번호 10번이라 사망했나?
# 번호와 생존은 상관없음

# 실무 에서는 아래를 제거
# 고유번호
# Primary Key
# ID
# UUID => 범용 고유 식별자(예)네트워크 카드의 맥주소)

# Name 컬럼

# 예시
# Braund, Mr. Owen Harris
# Cumings, Mrs. John Bradley

# 문제:
# 이름 종류가 너무 많음.
# 891명인데
# Name 고유값 ≈ 891개
# 거의 전부 다름.

# Braund → ?
# Owen → ?
# 그래서 # Mr # Mrs # Miss # Master 추출후 제거

# Ticket
# 예시

# A/5 21171
# PC 17599
# STON/O2 3101282

# 생존률과는 연관짓기가 힘듬
df_train["Ticket"].nunique()
# 681
# 891명 중
# 681개 종류
# 너무 많음.

# 분석 및 인사이트

# 컬럼	           제거 이유
# PassengerId	  단순 식별자
# Name	          고유값 과다
# Ticket	      종류 과다 + 의미 약함

df_train.drop(["PassengerId","Name","Ticket"],axis=1,inplace=True)
df_train

In [ ]:
# 피처 엔지니어링
# 피처 엔지니어링(Feature Engineering) 은 데이터 분석/머신러닝에서 가장 중요한 단계 중 하나임
# 한 줄 정의부터:
# 기존 데이터를 더 의미 있는 변수(Feature)로 변환하거나 새로 만드는 작업

# 원본 데이터
# ↓
# 모델이 이해하기 쉬운 정보로 가공

# 피처엔지니어링 이해하기

# | 종류    | 예시        |
# | ----- | --------- |
# | 결합    | 키+몸무게→BMI |
# | 분할    | 날짜→연·월·요일 |
# | 구간화   | 나이→연령대    |
# | 문자 추출 | 이름→호칭     |
# | 집계    | 구매금액→평균   |
# | 변환    | 로그 변환     |

# 전처리
# → 데이터를 깨끗하게 만드는 작업

# 피처 엔지니어링
# → 데이터를 똑똑하게 만드는 작업

df_train

# FamilySize 생성
# 원본
# sister(형제) + brother(자매) + Spouse(배우자)
# Parent + child 
# SibSp	Parch
# 1	      0
# 2	      1

# | 상황          | SibSp |
# | -----------   | ----: |
# | 혼자 탑승       |     0 |
# | 배우자와 탑승     |     1 |
# | 형 1명과 탑승    |     1 |
# | 형제 2명 + 배우자 |     3 |

# | 상황            | Parch |
# | ------------- | ----: |
# | 혼자            |     0 |
# | 부모 1명         |     1 |
# | 자녀 2명         |     2 |
# | 부모 2명 + 자녀 1명 |     3 |

# 2-5) 재구조화
# FamilySize 생성
# 탑승 가족수, 재구조화

df_train["FamilySize"] = df_train["SibSp"] + df_train["Parch"] + 1

df_train


In [ ]:
# 2-6)스케일링
# vi) 스케일링
# 1. 스케일링이란?
# 한 줄 정의:
# 서로 다른 크기의 숫자를 비슷한 기준으로 맞추는 것
# 머신러닝 딥러닝 에서 주로 수행
# 데이터 분석시에서 스케일링이 필요 없을수도 있음

# | Age |  Fare |
# | --: | ----: |
# |  22 |  7.25 |
# |  38 | 71.28 |
# |  35 | 53.10 |

# 문제:

# Age   → 0~80
# Fare  → 0~500

# 크기 차이가 큼.
# | 모델      | 스케일링 |
# | ------- | ---- |
# | 로지스틱 회귀 | 추천   |
# | KNN     | 필수   |
# | SVM     | 필수   |
# | 신경망     | 추천   |
# | 트리      | 불필요  |
# | 랜덤포레스트  | 불필요  |
